<a href="https://colab.research.google.com/github/vadapallimaneesha/machine_learning-/blob/main/week6portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install PySpark

*   This cell installs the `pyspark` library, which is essential for running Spark applications.

In [ ]:
!pip install pyspark

### Initialize Spark Session and Load Data

*   **Imports**: Imports `SparkSession`, `fetch_20newsgroups`, and `pandas`.
*   **Spark Session Management**: Stops any existing Spark session and creates a new one with increased memory.
*   **Data Loading**: Fetches the 20 Newsgroups dataset.
*   **DataFrame Creation**: Converts the dataset into a PySpark DataFrame.
*   **Data Overview**: Prints total documents, categories, and category distribution.
*   **Data Sampling**: Samples 25% of documents from each category for efficiency.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Start Spark Session
# spark = SparkSession.builder.appName("DocumentClassification").getOrCreate()

# Stop any existing Spark Session before creating a new one, handling potential connection errors
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception as e:
    print(f"Attempt to stop existing Spark session failed: {e}. Assuming session is dead and proceeding.")
    spark = None # Ensure spark is None if stopping failed


# Create a Spark session with increased memory to handle larger datasets
try:
    spark = SparkSession.builder.appName("DocumentClassificationTFIDF") \
        .config("spark.driver.memory", "8g") \
        .config("spark.driver.maxResultSize", "4g") \
        .getOrCreate()
except Exception as e:
    print(f"Failed to create Spark Session: {e}. This often means the Spark backend is not running or crashed. Please restart the Colab runtime (Runtime > Restart runtime...) and try again.")
    spark = None # Set spark to None if session creation fails

# Only proceed if Spark session was successfully created
if spark is not None:
    spark

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

# Show some details about the data
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Display the distribution of categories
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)
# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

# Show the number of documents after sampling
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")


Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20
Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910
Total number of documents after sampling: 4773


### TF-IDF with Logistic Regression Pipeline

*   **Tokenizer**: Splits text into words.
*   **HashingTF**: Converts words into numerical feature vectors.
*   **IDF**: Weights words based on their importance across documents.
*   **StringIndexer**: Converts text categories to numerical labels.
*   **Logistic Regression**: Defines the classification model.
*   **Pipeline**: Chains all preprocessing and model steps.
*   **Data Split**: Divides data into training and testing sets.
*   **Model Training**: Trains the pipeline model.
*   **Predictions**: Makes predictions on test data.
*   **Evaluation**: Calculates and prints the model's accuracy.

In [ ]:
# Prepare for Document Classification

# Step 1: Tokenize the text
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Step 2: Apply HashingTF
hashingTF = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=1000)

# Step 3: Compute IDF (Inverse Document Frequency)
idf = IDF(inputCol="raw_features", outputCol="features")

# Step 4: Convert category labels to numerical labels
indexer = StringIndexer(inputCol="category", outputCol="label")

# Step 5: Define the classifier (Logistic Regression in this case)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Set up the pipeline with all the stages
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, indexer, lr])

# Split the data into training and testing sets (80% train, 20% test)
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

# Step 6: Train the model using the pipeline
model = pipeline.fit(train_data)

# Step 7: Make predictions on the test data
predictions = model.transform(test_data)

# Show some of the predictions
predictions.select("text", "category", "prediction").show(5, truncate=False)

# Step 8: Evaluate the model's accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

# Display the accuracy
print(f"Model Accuracy: {accuracy:.2f}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Create Term-Document Matrix (TDM)

*   **Process Data**: Applies the trained model's transformations to get TF-IDF features.
*   **Extract Features**: Converts features into an RDD.
*   **NumPy Conversion**: Converts the RDD to a NumPy array.
*   **Pandas DataFrame**: Transforms the NumPy array into a Pandas DataFrame, representing the TDM.
*   **Display TDM**: Prints and displays the head of the Term-Document Matrix.

In [ ]:
import numpy as np
# Apply the pipeline to the sampled data (df_sampled) to get the 'features' column (TF-IDF vectors)
processed_data = model.transform(df_sampled)

# Extract the "features" column as an RDD
tdm_rdd = processed_data.select("features").rdd.map(lambda row: row[0])

# Convert the RDD of vectors into a numpy array
tdm_array = np.array(tdm_rdd.collect())

# Convert the numpy array into a DataFrame (this is our Term-Document Matrix)
tdm_df = pd.DataFrame(tdm_array)

# Show the Term-Document Matrix
print("Term-Document Matrix (TDM):")
print(tdm_df)

# Optional: Display the first few rows of the TDM
print(tdm_df.head())

Term-Document Matrix (TDM):
           0         1        2    3         4    5        6         7    \
0     0.000000  0.000000  1.90774  0.0  0.000000  0.0  0.00000  0.000000   
1     0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
2     0.000000  0.000000  0.00000  0.0  2.223593  0.0  2.70702  0.000000   
3     1.983452  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4     0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
...        ...       ...      ...  ...       ...  ...      ...       ...   
4768  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4769  1.983452  0.000000  0.00000  0.0  2.223593  0.0  0.00000  0.000000   
4770  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4771  0.000000  0.000000  0.00000  0.0  0.000000  0.0  0.00000  0.000000   
4772  0.000000  2.504946  0.00000  0.0  0.000000  0.0  0.00000  1.840601   

           8         9    ...       990  991       992  993

### Install PySpark and NLTK

*   This cell installs `pyspark` and `nltk` libraries, necessary for advanced text processing.

In [ ]:
!pip install pyspark nltk


### Download NLTK Data

*   Downloads 'wordnet' and 'omw-1.4' corpora, which are required for lemmatization.

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

### Import PySpark and NLTK Modules

*   Imports PySpark modules for Spark operations, ML features, and classification.
*   Imports NLTK modules like `PorterStemmer` and `WordNetLemmatizer` for text normalization functions.

In [ ]:
# Import necessary PySpark modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Import NLP Libraries for Stemming & Lemmatization
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import PorterStemmer, WordNetLemmatizer

### Define UDFs for Stemming and Lemmatization

*   Initializes `PorterStemmer` and `WordNetLemmatizer`.
*   Defines Python functions `stem_words` and `lemmatize_words`.
*   Converts these Python functions into PySpark User-Defined Functions (UDFs) for DataFrame application.

In [ ]:
#Define UDFs for Stemming & Lemmatization
# Initialize Stemmer and Lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define UDF for Stemming
def stem_words(words):
    return [stemmer.stem(word) for word in words]

# Define UDF for Lemmatization
def lemmatize_words(words):
    return [lemmatizer.lemmatize(word) for word in words]

# Convert Python functions to PySpark UDFs
stem_udf = udf(stem_words, ArrayType(StringType()))
lemma_udf = udf(lemmatize_words, ArrayType(StringType()))


### Re-initialize Spark Session

*   Safely stops any existing Spark session.
*   Creates a new Spark session with configured memory settings, ensuring a clean and properly resourced environment.

In [ ]:
# Stop any existing Spark Session before creating a new one, handling potential connection errors
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception as e:
    print(f"Attempt to stop existing Spark session failed: {e}. Assuming session is dead and proceeding.")
    spark = None # Ensure spark is None if stopping failed


# Create a Spark session with increased memory to handle larger datasets
try:
    spark = SparkSession.builder.appName("DocumentClassificationTFIDF") \
        .config("spark.driver.memory", "8g") \
        .config("spark.driver.maxResultSize", "4g") \
        .getOrCreate()
except Exception as e:
    print(f"Failed to create Spark Session: {e}. This often means the Spark backend is not running or crashed. Please restart the Colab runtime (Runtime > Restart runtime...) and try again.")
    spark = None # Set spark to None if session creation fails

# Only proceed if Spark session was successfully created
if spark is not None:
    spark

### Load Data for Preprocessing

*   Re-imports necessary libraries.
*   Fetches the 20 Newsgroups dataset again.
*   Converts the data into a PySpark DataFrame.
*   Prints data details like total documents, categories, and their distribution.
*   Samples 25% of the documents for efficient processing.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

# Show some details about the data
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Display the distribution of categories
category_counts = df.groupBy('category').count().toPandas()
print("Category distribution before filtering (25%):")
print(category_counts)

# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

# Show the number of documents after sampling
total_documents_after_sampling = df_sampled.count()
print(f"Total number of documents after sampling: {total_documents_after_sampling}")


Total number of documents: 18846
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Number of categories: 20
Category distribution before filtering (25%):
    category  count
0         19    628
1          0    799
2          7    990
3          6    975
4          9    994
5         17    940
6          5    988
7          1    973
8         10    999
9          3    982
10        12    984
11         8    996
12        11    991
13         2    985
14         4    963
15        13    990
16        18    775
17        14    987
18        15    997
19        16    910
Total number of documents after sampling: 4773


### Text Preprocessing: Tokenization, Stopword Removal, Stemming, and Lemmatization

*   **Tokenization**: Splits text into `words`.
*   **Stopword Removal**: Removes common words, creating `filtered_words`.
*   **Stemming**: Applies `stem_udf` to create `stemmed_words`.
*   **Lemmatization**: Applies `lemma_udf` to create `lemmatized_words`.
*   **Show Results**: Displays samples of original, filtered, stemmed, and lemmatized words.

In [ ]:
# Tokenization
tokenizer = Tokenizer(inputCol="text", outputCol="words")
df = tokenizer.transform(df)

# Stopword Removal
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
df = remover.transform(df)

# Apply Stemming
df = df.withColumn("stemmed_words", stem_udf(col("filtered_words")))

# Apply Lemmatization
df = df.withColumn("lemmatized_words", lemma_udf(col("filtered_words")))

# Show Results
df.select("text", "filtered_words", "stemmed_words", "lemmatized_words").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Compute TF-IDF After Preprocessing

*   **HashingTF**: Converts `lemmatized_words` into `raw_features` (numerical vectors).
*   **IDF**: Computes and applies Inverse Document Frequency to create final TF-IDF `features`.
*   **Show Features**: Displays text and its corresponding TF-IDF feature vectors.

In [ ]:

# Compute TF-IDF After Preprocessing
# Apply HashingTF to the lemmatized words
hashingTF = HashingTF(inputCol="lemmatized_words", outputCol="raw_features", numFeatures=500)
df = hashingTF.transform(df)

# Compute IDF
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(df)
df = idf_model.transform(df)

# Show TF-IDF Features
df.select("text", "features").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Train and Evaluate Logistic Regression Classifier with TF-IDF Features

*   **StringIndexer**: Converts categories to numerical labels.
*   **Data Split**: Divides data into training and testing sets.
*   **Model Training**: Trains a `LogisticRegression` model.
*   **Predictions**: Generates predictions on test data.
*   **Evaluation**: Calculates and prints the accuracy of the TF-IDF model.

In [ ]:
import pyspark.sql.functions as F

#Step 5: Train & Evaluate Logistic Regression Classifier

# Convert category labels to numerical labels
indexer = StringIndexer(inputCol="category", outputCol="label")

# Drop 'label' column if it already exists to allow re-running the cell
if "label" in df.columns:
    df = df.drop("label")

df = indexer.fit(df).transform(df)
df.select("category", "label").distinct().show()
# Split Data
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Train Logistic Regression Model
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)

# Predictions
predictions = lr_model.transform(test_data)
predictions.select("text", "category", "prediction").show(truncate=False)

# Evaluate Model Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy_tf_idf = evaluator.evaluate(predictions)
print(f"TF-IDF Model Accuracy: {accuracy_tf_idf:.2f}")

+--------+-----+
|category|label|
+--------+-----+
|       2|  9.0|
|      15|  1.0|
|      19| 19.0|
|       3| 11.0|
|       8|  2.0|
|       0| 17.0|
|       6| 12.0|
|       7|  6.0|
|      16| 16.0|
|      14|  8.0|
|      13|  5.0|
|      17| 15.0|
|      11|  4.0|
|       1| 13.0|
|       4| 14.0|
|       5|  7.0|
|      18| 18.0|
|      12| 10.0|
|      10|  0.0|
|       9|  3.0|
+--------+-----+

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Display Category-Label Mapping

*   Displays the unique mappings between the original `category` names and their newly assigned numerical `label` IDs.

In [ ]:

df.select("category", "label").distinct().show()

+--------+-----+
|category|label|
+--------+-----+
|       2|  9.0|
|      15|  1.0|
|      19| 19.0|
|       3| 11.0|
|       8|  2.0|
|       0| 17.0|
|       6| 12.0|
|       7|  6.0|
|      16| 16.0|
|      14|  8.0|
|      13|  5.0|
|      17| 15.0|
|      11|  4.0|
|       1| 13.0|
|       4| 14.0|
|       5|  7.0|
|      18| 18.0|
|      12| 10.0|
|      10|  0.0|
|       9|  3.0|
+--------+-----+



### Train Word2Vec Model

*   **Import Word2Vec**: Imports the `Word2Vec` module.
*   **Model Initialization**: Configures `Word2Vec` with `vectorSize` and `inputCol`.
*   **Model Training**: Fits the `Word2Vec` model on the data.
*   **Feature Transformation**: Transforms the DataFrame to add `featuresW2Vector` (word embeddings).
*   **Show Features**: Displays text and its Word2Vec feature vectors.

In [ ]:
#Train Word2Vec Model
#import Word2Vec
from pyspark.ml.feature import  Word2Vec   # Import Word2Vec here
word2Vec = Word2Vec(vectorSize=100, minCount=1, inputCol="lemmatized_words", outputCol="featuresW2Vector")
word2Vec_model = word2Vec.fit(df)
df = word2Vec_model.transform(df)

df.select("text", "featuresW2Vector").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Train and Evaluate Logistic Regression Classifier on Word2Vec Features

*   **Data Split**: Divides data (with Word2Vec features) into training and testing sets.
*   **Model Training**: Trains a `LogisticRegression` model using Word2Vec features.
*   **Predictions**: Generates predictions on the test data.
*   **Evaluation**: Calculates and prints the accuracy of the Word2Vec model.

In [ ]:
#Step 7: Train & Evaluate Logistic Regression Classifier on Word2Vec Features
# Split Data
train_data_w2v, test_data_w2v = df.randomSplit([0.8, 0.2], seed=42)

# Train Model
lr_w2v = LogisticRegression(featuresCol="featuresW2Vector", labelCol="label")
lr_w2v_model = lr_w2v.fit(train_data_w2v)

# Predictions
predictions_w2v = lr_w2v_model.transform(test_data_w2v)
predictions_w2v.select("text", "category", "prediction").show(truncate=False)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Evaluate Model Accuracy
accuracy_w2v = evaluator.evaluate(predictions_w2v)
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.2f}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------